Ingest NASA NeoWs, JPL Sentry, and SBDB CAD API endpoints into static JSON datasets.


In [1]:
import os
import json
import urllib.parse
import urllib.request
from pathlib import Path

# Resolve project root & dataset directory paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"

# Load API Key from .env if present
def get_api_key():
    env_file = PROJECT_ROOT / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
    return os.getenv("NASA_API_KEY") or os.getenv("API_KEY") or "DEMO_KEY"

API_KEY = get_api_key()
print(f"Project root : {PROJECT_ROOT}")
print(f"API Key      : {'Custom Key (Active)' if API_KEY != 'DEMO_KEY' else 'DEMO_KEY'}")

Project root : /home/ahmad/projects/spaceapp
API Key      : Custom Key (Active)


Fetch NeoWs asteroid browse and 7-day feed data.


In [2]:
neows_dir = DATASETS_DIR / "nasa_neows"
neows_dir.mkdir(parents=True, exist_ok=True)

def fetch_json(url, dest_path):
    req = urllib.request.Request(url, headers={"User-Agent": "spaceapp/1.0"})
    with urllib.request.urlopen(req, timeout=30) as res:
        data = json.loads(res.read().decode("utf-8"))
    with open(dest_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    return data

# 1. Browse Asteroid Data
browse_url = f"https://api.nasa.gov/neo/rest/v1/neo/browse?api_key={API_KEY}"
browse_data = fetch_json(browse_url, neows_dir / "browse.json")
print(f"[A1] Browse saved: {neows_dir / 'browse.json'} ({len(browse_data.get('near_earth_objects', []))} objects on page)")

# 2. Daily Asteroid Feed (2026-10-25 to 2026-10-31)
params = urllib.parse.urlencode({
    "start_date": "2026-10-25",
    "end_date": "2026-10-31",
    "api_key": API_KEY
})
feed_url = f"https://api.nasa.gov/neo/rest/v1/feed?{params}"
feed_data = fetch_json(feed_url, neows_dir / "feed.json")
print(f"[A2] Feed saved  : {neows_dir / 'feed.json'} ({feed_data.get('element_count')} total elements)")

[A1] Browse saved: /home/ahmad/projects/spaceapp/datasets/nasa_neows/browse.json (20 objects on page)
[A2] Feed saved  : /home/ahmad/projects/spaceapp/datasets/nasa_neows/feed.json (44 total elements)


Fetch JPL Sentry impact risk monitoring data.


In [3]:
sentry_dir = DATASETS_DIR / "cneos_sentry"
sentry_dir.mkdir(parents=True, exist_ok=True)

sentry_url = "https://ssd-api.jpl.nasa.gov/sentry.api"
sentry_data = fetch_json(sentry_url, sentry_dir / "sentry.json")
print(f"[B] Sentry saved : {sentry_dir / 'sentry.json'} ({sentry_data.get('count')} tracked objects)")

[B] Sentry saved : /home/ahmad/projects/spaceapp/datasets/cneos_sentry/sentry.json (2251 tracked objects)


Fetch JPL SBDB close-approach data filtered by dist <= 10 LD.


In [4]:
cad_dir = DATASETS_DIR / "sbdb_cad"
cad_dir.mkdir(parents=True, exist_ok=True)

cad_params = urllib.parse.urlencode({
    "dist-max": "10LD",
    "date-min": "2026-01-01",
    "sort": "dist"
})
cad_url = f"https://ssd-api.jpl.nasa.gov/cad.api?{cad_params}"
cad_data = fetch_json(cad_url, cad_dir / "cad.json")
print(f"[C] CAD saved    : {cad_dir / 'cad.json'} ({cad_data.get('count')} approach events)")

[C] CAD saved    : /home/ahmad/projects/spaceapp/datasets/sbdb_cad/cad.json (1097 approach events)


Verify that all downloaded JSON dataset files exist and are valid.


In [5]:
targets = [
    neows_dir / "browse.json",
    neows_dir / "feed.json",
    sentry_dir / "sentry.json",
    cad_dir / "cad.json"
]

print("Dataset File Integrity Verification:")
for t in targets:
    assert t.exists() and t.stat().st_size > 0, f"Invalid file: {t}"
    size_kb = t.stat().st_size / 1024
    print(f"✓ {t.relative_to(PROJECT_ROOT)}: {size_kb:.1f} KB")

Dataset File Integrity Verification:
✓ datasets/nasa_neows/browse.json: 466.3 KB
✓ datasets/nasa_neows/feed.json: 77.4 KB
✓ datasets/cneos_sentry/sentry.json: 844.0 KB
✓ datasets/sbdb_cad/cad.json: 283.0 KB
